# RAG Pipeline — Fine-Tuned Qwen 2.5 Coder 7B

This notebook implements an **error-driven RAG pipeline** using a **fine-tuned** Qwen 2.5 Coder 7B model.

## Model
| Item | Value |
|------|-------|
| Fine-tuned model | `H4miid/qwen2_5_coder_7b_merged_f16.gguf` |
| Format | GGUF |
| Backend | llama.cpp server (OpenAI-compatible API) |

## Pipeline Overview
1. **Load Dataset** — train/test split
2. **Load Failed Samples** — from Pre-Test smoke report
3. **RAG Retrieval** — ChromaDB → BGE bi-encoder → cross-encoder reranking → **OpenRouter summarization (≤ 5 bullet hints)**
4. **LangChain Chain** — structured RAG prompt → llama.cpp server
5. **Evaluation** — syntax check + runtime smoke test

## Observability
- **LangSmith** traces every LLM call *and* every retrieval / reranking / summarization step.
```

## 1 — Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

# --- Standard library ---
import json, os, re, time, subprocess, shutil, sys
import tempfile
from pathlib import Path
from difflib import SequenceMatcher
from datetime import datetime

# --- Data & ML ---
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# --- Vector DB & Embeddings ---
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

# --- LangChain & OpenAI ---
import openai, requests
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAI as LangchainOpenAI

# --- HuggingFace & Env ---
from huggingface_hub import hf_hub_download, login
from dotenv import load_dotenv

# --- LangSmith tracing ---
try:
    from langsmith import traceable
    LANGSMITH_OK = True
except ImportError:
    # If langsmith is not installed, create a no-op decorator so code still runs
    LANGSMITH_OK = False
    def traceable(*args, **_kwargs):
        """Dummy decorator when langsmith is not installed."""
        def decorator(fn):
            return fn
        if args and callable(args[0]):
            return args[0]
        return decorator

# ── Load .env ────────────────────────────────────────────────────────────────
load_dotenv()

# HuggingFace login for private model repos
HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✓ Logged in to HuggingFace Hub")
else:
    print("⚠ HF_TOKEN not found — private model download may fail")

# ── LangSmith setup (enable tracing if key exists) ───────────────────────────
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY", "")
if LANGCHAIN_API_KEY:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_API_KEY"]    = LANGCHAIN_API_KEY
    os.environ["LANGCHAIN_PROJECT"]    = "MentorApp-RAG-SFT"
    print("✓ LangSmith tracing enabled  →  project: MentorApp-RAG-SFT")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("ℹ LangSmith tracing disabled (add LANGCHAIN_API_KEY to .env to enable)")

print("✓ All imports loaded")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Logged in to HuggingFace Hub
✓ LangSmith tracing enabled  →  project: MentorApp-RAG-SFT
✓ All imports loaded


## 0 — GPU Detection & llama.cpp Build

In [2]:
def check_gpu():
    """Return True if an NVIDIA GPU is available."""
    if shutil.which("nvidia-smi"):
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10,
        )
        if result.returncode == 0:
            for i, line in enumerate(result.stdout.strip().splitlines()):
                name, total, free = [p.strip() for p in line.split(",")]
                print(f"GPU {i}: {name}  |  VRAM: {int(total)/1024:.1f} GB total, {int(free)/1024:.1f} GB free")
            return True

    # Fallback: try PyTorch
    try:
        import torch
        if torch.cuda.is_available():
            print(f"GPU: {torch.cuda.get_device_name(0)}")
            return True
    except ImportError:
        pass

    print("⚠ No GPU detected — will run on CPU")
    return False

GPU_AVAILABLE = check_gpu()

GPU 0: NVIDIA GeForce RTX 5090  |  VRAM: 31.8 GB total, 31.4 GB free


In [8]:
# Clone llama.cpp and build it with CUDA support.
# This only needs to run once — subsequent runs skip the clone/build.

LLAMA_CPP_DIR = Path("../llama.cpp").resolve()

# Step 1: Clone the repo (skip if it already exists)
if not LLAMA_CPP_DIR.exists():
    print("Cloning llama.cpp …")
    subprocess.run(
        ["git", "clone", "https://github.com/ggerganov/llama.cpp.git", str(LLAMA_CPP_DIR)],
        check=True,
    )
    print(f"✓ Cloned → {LLAMA_CPP_DIR}")
else:
    print(f"✓ llama.cpp already at {LLAMA_CPP_DIR}")

# Step 2: CMake configure + build
build_dir = LLAMA_CPP_DIR / "build"
build_dir.mkdir(exist_ok=True)

print("Configuring with CUDA …")
subprocess.run(["cmake", "..", "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release"],
               cwd=str(build_dir), check=True)

print("Building (may take a few minutes) …")
subprocess.run(["cmake", "--build", ".", "--config", "Release", "-j"],
               cwd=str(build_dir), check=True)

# Step 3: Locate the server executable
candidates = [
    build_dir / "bin" / "Release" / "llama-server.exe",  # Windows MSVC
    build_dir / "bin" / "llama-server.exe",               # Windows Ninja
    build_dir / "bin" / "llama-server",                   # Linux / macOS
]
LLAMA_SERVER_EXE = next((p for p in candidates if p.exists()), None)

# Fallback: recursive search
if not LLAMA_SERVER_EXE:
    hits = [f for f in build_dir.rglob("llama-server*") if f.is_file() and f.suffix in ("", ".exe")]
    if hits:
        LLAMA_SERVER_EXE = hits[0]

if LLAMA_SERVER_EXE:
    print(f"✓ Server executable: {LLAMA_SERVER_EXE}")
else:
    raise FileNotFoundError("llama-server not found after build — check CMake output.")

✓ llama.cpp already at /workspace/MentorApp/llama.cpp
Configuring with CUDA …


CMAKE_BUILD_TYPE=Release


-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- CUDA Toolkit found
-- Replacing 120-real in CMAKE_CUDA_ARCHITECTURES_NATIVE with 120a-real
-- Using CMAKE_CUDA_ARCHITECTURES=120a-real CMAKE_CUDA_ARCHITECTURES_NATIVE=120a-real
-- CUDA host compiler is GNU 13.3.0
-- Including CUDA backend
-- ggml version: 0.9.7
-- ggml commit:  f5ddcd169
-- OpenSSL found: 3.0.13
-- Generating embedded license file for target: common
-- Configuring done (0.6s)
-- Generating done (0.3s)
-- Build files have been written to: /workspace/MentorApp/llama.cpp/build
Building (may take a few minutes) …
[  0%] Built target build_info
[  0%] Built target sha256
[  0%] Built target llama-minicpmv-cli
[  1%] Built target xxhash
[  1%] Built target llama-llava-cli
[  1%] Built target sha1
[  

## 2 — Configuration

In [56]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH       = Path("../Datasets/final_dataset.json").resolve()
CHROMA_DIR      = str(Path("../VectorDB/chroma_library_docs").resolve())
OUT_DIR         = Path("RAG_outputs/RAG_with_SFT").resolve()
MODEL_DIR       = Path("../Models").resolve()
COLLECTION_NAME = "library_docs"

SMOKE_REPORT_PATH = Path(
    r"../Fine-Tuning/Qwen/Fine-Tune_Results/fine_tuned_eval_runtime_A/smoke_report.json"
).resolve()

# Path to the SFT model's prediction outputs (contains prediction.py per sample)
SFT_EVAL_OUTPUTS_DIR = Path(
    "../Fine-Tuning/Qwen/Fine-Tune_Results/fine_tuned_eval_outputs_A/setting_A"
).resolve()

# ── Model ─────────────────────────────────────────────────────────────────────
HF_MODEL_REPO   = "H4miid/qwen2_5_coder_7b_merged_f16.gguf"
HF_MODEL_FILE   = "qwen2_5_coder_7b_merged_f16.gguf"
LOCAL_MODEL_PATH = MODEL_DIR / HF_MODEL_FILE

N_CTX        = 8192
N_GPU_LAYERS = -1 if GPU_AVAILABLE else 0   # -1 = offload everything to GPU
TEMPERATURE  = 0.0
MAX_TOKENS   = 2048   # reduced from 4096 — most code fixes need < 2048 tokens

# ── Dataset split ─────────────────────────────────────────────────────────────
SEED      = 42
TEST_SIZE = 0.15

# ── RAG retrieval settings ───────────────────────────────────────────────────
N_RETRIEVE         = 10    # bi-encoder candidates per library
N_RERANK           = 3     # top-k kept after cross-encoder
MAX_QUERY          = 500   # max chars sent as RAG query
MAX_CTX_CHARS      = 3000  # char budget for raw context
MIN_RERANKER_SCORE = 0.0   # score gate for cross-encoder (0.0 = keep all; filtering moved to summariser)
RERANKER_MODEL     = "BAAI/bge-reranker-base"

# ── Runtime eval ──────────────────────────────────────────────────────────────
TIMEOUT   = 300
FORCE_CPU = False

# ── llama.cpp server ──────────────────────────────────────────────────────────
LLAMA_PORT     = 8081
LLAMA_HOST     = "127.0.0.1"
LLAMA_BASE_URL = f"http://{LLAMA_HOST}:{LLAMA_PORT}"

# ── OpenRouter (doc summarization) ───────────────────────────────────────────
OPENROUTER_API_KEY  = os.getenv("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL    = "openai/gpt-oss-120b"

# Fallback models — tried in order if the primary model returns None
OPENROUTER_FALLBACK_MODELS = [
    "openai/gpt-4.1-mini",
    "qwen/qwen3-8b",
    "mistralai/ministral-14b-2512",
]

# ── Summarization tuning ─────────────────────────────────────────────────────
SUMMARIZE_MAX_RETRIES   = 3        # retries per model before trying the next
SUMMARIZE_RETRY_DELAY   = 3        # initial delay in seconds (doubles each retry)
SUMMARIZE_THROTTLE      = 1.5      # minimum seconds between summarization API calls
SUMMARIZE_MAX_DOC_CHARS = 1500     # trim docs sent for summarization (less input → more reliable output)

# ── Create dirs ───────────────────────────────────────────────────────────────
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset         : {DATA_PATH}")
print(f"ChromaDB        : {CHROMA_DIR}")
print(f"Output dir      : {OUT_DIR}")
print(f"SFT eval outputs: {SFT_EVAL_OUTPUTS_DIR}")
print(f"Model           : {HF_MODEL_REPO}")
print(f"GPU layers      : {N_GPU_LAYERS}")
print(f"llama.cpp server: {LLAMA_BASE_URL}")
print(f"OpenRouter model: {OPENROUTER_MODEL}")
print(f"Fallback models : {OPENROUTER_FALLBACK_MODELS}")

Dataset         : /workspace/MentorApp/Datasets/final_dataset.json
ChromaDB        : /workspace/MentorApp/VectorDB/chroma_library_docs
Output dir      : /workspace/MentorApp/RAG_Pipelines/RAG_outputs/RAG_with_SFT
SFT eval outputs: /workspace/MentorApp/Fine-Tuning/Qwen/Fine-Tune_Results/fine_tuned_eval_outputs_A/setting_A
Model           : H4miid/qwen2_5_coder_7b_merged_f16.gguf
GPU layers      : -1
llama.cpp server: http://127.0.0.1:8081
OpenRouter model: openai/gpt-oss-120b
Fallback models : ['openai/gpt-4.1-mini', 'qwen/qwen3-8b', 'mistralai/ministral-14b-2512']


## 3 — Load Dataset & Split

In [4]:
# Load the full dataset
with open(DATA_PATH, "r", encoding="utf-8") as f:
    dataset_full = json.load(f)

print(f"Total samples: {len(dataset_full)}")

# Generate a synthetic ID for each sample from its index + title
def _make_sid(idx: int, title: str) -> str:
    """Create a short filesystem-safe identifier like '000_Adult_Income_Hyperparameter_Grid'."""
    slug = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_")
    return f"{idx:03d}_{slug}"

for i, sample in enumerate(dataset_full):
    sample["_sid"] = _make_sid(i, sample.get("title", f"sample_{i}"))
    sample["_orig_idx"] = i          # keep original 0-based index

# Train/test split
train_set, test_set = train_test_split(
    dataset_full,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)

print(f"Train set: {len(train_set)} samples")
print(f"Test set : {len(test_set)} samples")
print(f"Example SID: {dataset_full[0]['_sid']}")

Total samples: 582
Train set: 494 samples
Test set : 88 samples
Example SID: 000_Adult_Income_Hyperparameter_Grid


## 4 — Load Smoke Report & Identify Failed Samples

We only apply RAG to samples that **failed runtime** in the SFT pre-test. These are the samples the fine-tuned model struggled with and may benefit from retrieval augmentation.

In [5]:
def clean_traceback(text: str) -> str:
    """Clean raw stderr text for use in LLM prompts.

    Fixes applied:
    - Rejoin lines hard-wrapped by the 80-column terminal capture
    - Remove ^^^ / ~~~ caret indicator lines  (Python 3.13+)
    - Remove  ...<N lines>...  folding markers
    - Strip DeprecationWarning / UserWarning / FutureWarning noise
    - Shorten absolute file paths to just the filename
    - Collapse excessive blank lines
    """
    if not text or not text.strip():
        return ""

    lines = text.split("\n")

    # ── 1. Rejoin hard-wrapped lines ──────────────────────────────────────
    # The subprocess captured stderr from a console with ~80-col width.
    # Lines that got truncated mid-word are rejoined here.
    joined: list[str] = []
    for line in lines:
        stripped = line.rstrip()
        if (
            joined
            and stripped
            # Continuation lines do NOT start with whitespace or a new
            # traceback entry; they are leftover fragments from wrapping.
            and not re.match(r'^(\s|Traceback |File "|[A-Z]\w*(Error|Warning|Exception))', stripped)
            and not stripped.startswith(("---", "==="))
            # Previous line was "long enough" to have been wrapped
            and len(joined[-1]) >= 60
        ):
            joined[-1] = joined[-1].rstrip() + stripped
        else:
            joined.append(stripped)

    # ── 2. Remove noisy lines ─────────────────────────────────────────────
    cleaned: list[str] = []
    skip_next = False
    for line in joined:
        # Caret / tilde indicator lines  (e.g. "    ^^^^^^^^^^^")
        if re.match(r'^\s*[\^~]+\s*$', line):
            continue
        # Folding markers  (e.g. "    ...<5 lines>...")
        if re.match(r'^\s*\.\.\.<\d+ lines>\.\.\.\s*$', line):
            continue
        # Warning lines (DeprecationWarning, UserWarning, etc.)
        if re.search(r'(DeprecationWarning|UserWarning|FutureWarning|VisibleDeprecationWarning):', line):
            skip_next = True          # also skip the source line that follows
            continue
        if skip_next:
            skip_next = False
            # If the next line is a new traceback or error, keep it
            if not re.match(r'^(Traceback |  File "|[A-Z]\w*(Error|Exception))', line.strip()):
                continue
        cleaned.append(line)

    # ── 3. Shorten absolute paths to filename ─────────────────────────────
    text_out = "\n".join(cleaned)
    # C:\...\file.py  or  c:\...\file.py  →  file.py
    text_out = re.sub(r'[A-Za-z]:\\(?:[^\\"\s:]+\\)*([^\\"\s:]+\.py)', r'\1', text_out)

    # ── 4. Collapse multiple blank lines ──────────────────────────────────
    text_out = re.sub(r'\n{3,}', '\n\n', text_out).strip()

    return text_out


# Quick test
_test_stderr = (
    "c:\\Users\\hbahmanyar\\MentorApp\\.venv\\Lib\\site-packages\\numpy\\lib\\_format_impl.py:\n"
    "838: VisibleDeprecationWarning: dtype(): align issue\n"
    "  array = pickle.load(fp)\n"
    "Traceback (most recent call last):\n"
    "  File \"C:\\Users\\hbahmanyar\\MentorApp\\test.py\", line 10, in <module>\n"
    "    ~~~~~~~~~~~^^^^^^^^^^\n"
    "ValueError: Input X contains NaN.\n"
)
print("Clean traceback test:")
print(clean_traceback(_test_stderr))

Clean traceback test:
Traceback (most recent call last):
  File "test.py", line 10, in <module>
ValueError: Input X contains NaN.


In [6]:
# Load the smoke report from SFT pre-test
with open(SMOKE_REPORT_PATH, "r", encoding="utf-8") as f:
    smoke_report = json.load(f)

print(f"Loaded smoke report with {len(smoke_report)} entries\n")

# NOTE: The smoke report was generated on Windows, so paths use backslashes.
# We use PureWindowsPath to parse them correctly on any OS.
from pathlib import PureWindowsPath

# ── Build mapping from eval folder → smoke report entry ──────────────────────
# The eval folder names (000_Foo, 001_Bar, …) follow the SFT eval's own
# sequential numbering — NOT the original dataset index.  We use the folder
# name to look up both prediction.py AND correct.py from the eval outputs,
# so the mapping is always consistent.
smoke_by_folder = {}
for entry in smoke_report:
    eval_folder = PureWindowsPath(entry["file"]).parent.name
    passed = entry.get("ok", False) and not entry.get("skipped", False)
    smoke_by_folder[eval_folder] = {**entry, "_passed": passed, "_eval_folder": eval_folder}

# ── Build rag_samples: failed samples with SFT prediction + runtime error ────
# For each failed sample we:
#   1. Read the SFT model's prediction.py from fine_tuned_eval_outputs_A  (= buggy code for RAG)
#   2. Use stderr_tail from the smoke report                             (= runtime error message)
#   3. Read correct.py from the SAME eval folder                         (= reference for similarity)

failed_entries = {
    folder: info
    for folder, info in smoke_by_folder.items()
    if not info["_passed"]
}
print(f"Failed samples in pre-test: {len(failed_entries)}")

# Also keep smoke_by_orig_idx for the SFT-vs-RAG comparison cell later
smoke_by_orig_idx = {}

rag_samples = []
skipped_timeout = 0
for eval_folder, info in sorted(failed_entries.items()):
    prediction_path = SFT_EVAL_OUTPUTS_DIR / eval_folder / "prediction.py"
    correct_path    = SFT_EVAL_OUTPUTS_DIR / eval_folder / "correct.py"

    if not prediction_path.exists():
        print(f"  ⚠ SKIP {eval_folder}: prediction.py not found at {prediction_path}")
        continue

    stderr_error = info.get("stderr_tail", "")

    # Skip samples whose only failure was a timeout — RAG cannot fix those
    if stderr_error.strip().startswith("TIMEOUT"):
        skipped_timeout += 1
        print(f"  ⏭ SKIP {eval_folder}: timeout (not a code bug)")
        continue

    sft_prediction = prediction_path.read_text(encoding="utf-8")
    correct_code   = correct_path.read_text(encoding="utf-8") if correct_path.exists() else ""

    # Build a SID from the eval folder name for output naming
    sid = eval_folder

    rag_samples.append({
        "_eval_folder":  eval_folder,
        "_sid":          sid,
        "title":         re.sub(r"^\d+_", "", eval_folder).replace("_", " "),
        "correct_code":  correct_code,
        # ── These come from the SFT eval, NOT the original dataset ────
        "sft_prediction": sft_prediction,       # the code the SFT model produced (failed)
        "runtime_error":  clean_traceback(stderr_error),  # cleaned stderr
    })

# Populate smoke_by_orig_idx for all entries (used by comparison cell)
# Use eval_folder index as a proxy for orig_idx
for folder, info in smoke_by_folder.items():
    idx = int(folder.split("_")[0])  # e.g. "026_Foo" → 26
    smoke_by_orig_idx[idx] = info

print(f"\nSkipped (timeout): {skipped_timeout}")
print(f"Samples to process with RAG: {len(rag_samples)}")
for s in rag_samples[:10]:
    has_correct = "✓" if s["correct_code"] else "✗"
    print(f"  - {s['_eval_folder']}  (correct.py: {has_correct})")
if len(rag_samples) > 10:
    print(f"  ... and {len(rag_samples) - 10} more")

Loaded smoke report with 88 entries

Failed samples in pre-test: 18
  ⏭ SKIP 014_IMDB_Sentiment_Analysis_using_LSTM: timeout (not a code bug)
  ⏭ SKIP 080_IMDB_Sentiment_Bidirectional_LSTM_Embedd: timeout (not a code bug)

Skipped (timeout): 2
Samples to process with RAG: 16
  - 016_IMDB_Sentiment_Analysis_Naive_Bayes  (correct.py: ✓)
  - 017_Reuters_News_Topic_Classification  (correct.py: ✓)
  - 025_Diabetes_Progression_LightGBM_Regression  (correct.py: ✓)
  - 026_Titanic_Survival_ROC_Curve  (correct.py: ✓)
  - 033_Digits_Autoencoder_Reconstruction  (correct.py: ✓)
  - 046_Penguins_Migration_Time_Series_ARIMA  (correct.py: ✓)
  - 047_Heart_Model_Calibration_Plot  (correct.py: ✓)
  - 048_Sunspots_SARIMAX_Seasonal_Forecast_Model  (correct.py: ✓)
  - 057_MNIST_Voting_Ensemble_Classification  (correct.py: ✓)
  - 058_Diabetes_Progression_Neural_Network_Regr  (correct.py: ✓)
  ... and 6 more


## 5 — Download Model & Start llama.cpp Server

Download the GGUF model from HuggingFace Hub, then start the **llama.cpp server** to serve it via an OpenAI-compatible API on `localhost`.

In [10]:
# Download the GGUF model from HuggingFace Hub if not already present
if not LOCAL_MODEL_PATH.exists():
    print(f"Downloading model from {HF_MODEL_REPO}...")
    downloaded_path = hf_hub_download(
        repo_id=HF_MODEL_REPO,
        filename=HF_MODEL_FILE,
        local_dir=str(MODEL_DIR),
        local_dir_use_symlinks=False
    )
    print(f"Downloaded to: {downloaded_path}")
else:
    print(f"Model already exists at: {LOCAL_MODEL_PATH}")

# Verify model file size
model_size_gb = LOCAL_MODEL_PATH.stat().st_size / (1024**3)
print(f"Model size: {model_size_gb:.2f} GB")

Downloaded to: /workspace/MentorApp/Models/qwen2_5_coder_7b_merged_f16.gguf
Model size: 14.19 GB


In [57]:
# Start the llama.cpp HTTP server (runs in the background).
# Then create a LangChain LLM client that talks to it.

print(f"Starting llama.cpp server on {LLAMA_BASE_URL} …")

server_process = subprocess.Popen(
    [str(LLAMA_SERVER_EXE),
     "-m",    str(LOCAL_MODEL_PATH),
     "--port", str(LLAMA_PORT),
     "--host", LLAMA_HOST,
     "-ngl",  str(N_GPU_LAYERS),
     "-c",    str(N_CTX)                       # flash attention — faster for large contexts
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait until the server reports healthy 
for sec in range(120):
    try:
        if requests.get(f"{LLAMA_BASE_URL}/health", timeout=2).status_code == 200:
            print(f"✓ Server ready ({sec + 1}s)")
            break
    except requests.ConnectionError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("llama.cpp server did not start within 120 s")

# LangChain LLM — points at the local server's OpenAI-compatible API
llm = LangchainOpenAI(
    base_url=f"{LLAMA_BASE_URL}/v1",
    api_key="not-needed",            # server has no auth
    model=LOCAL_MODEL_PATH.stem,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

# Smoke test
_ = llm.invoke("Say hello in one word.")
print("✓ LLM ready")

Starting llama.cpp server on http://127.0.0.1:8081 …
✓ Server ready (4s)
✓ LLM ready


## 6 — Load Vector Store & Reranker

Connect to the pre-built ChromaDB containing library documentation embeddings, and initialize the cross-encoder reranker for result refinement.

In [11]:
# Connect to ChromaDB
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

print(f"✓ Connected to ChromaDB collection: {COLLECTION_NAME}")
print(f"  Documents in collection: {collection.count()}")

# Initialize embedding model (same as used for indexing)
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")
print(f"✓ Embedding model loaded: BAAI/bge-base-en-v1.5")

# Initialize cross-encoder reranker
reranker = CrossEncoder(RERANKER_MODEL)
print(f"✓ Reranker loaded: {RERANKER_MODEL}")

✓ Connected to ChromaDB collection: library_docs
  Documents in collection: 647


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4678.47it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Embedding model loaded: BAAI/bge-base-en-v1.5


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4779.60it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Reranker loaded: BAAI/bge-reranker-base


## 7 — RAG Helper Functions

Three retrieval utilities (each traced by **LangSmith** when enabled):

| Function | What it does |
|----------|-------------|
| `retrieve_for_library` | Query ChromaDB for docs matching the error |
| `rerank_and_filter` | Cross-encoder reranking + score gate |
| `build_rag_context` | Orchestrate retrieval across libraries |

In [12]:
def extract_error_signal(error_text: str) -> str:
    """Extract the core error message from a traceback for a focused RAG query.

    Pulls the final 'SomeError: description' line and, if present, the
    offending source line.  Falls back to the last MAX_QUERY chars.
    """
    lines = [l.strip() for l in error_text.strip().splitlines() if l.strip()]
    # Find the last line matching "SomeError: ..." or "SomeException: ..."
    error_line = ""
    source_line = ""
    for i, line in enumerate(lines):
        if re.match(r"^\w*(Error|Exception):", line):
            error_line = line
            # The previous non-empty line is often the offending source
            if i > 0 and not lines[i - 1].startswith(("File ", "Traceback")):
                source_line = lines[i - 1]
    if error_line:
        parts = [error_line]
        if source_line:
            parts.insert(0, source_line)
        return "\n".join(parts)
    # Fallback: last portion of the traceback
    return error_text[-MAX_QUERY:]


@traceable(name="retrieve_for_library")
def retrieve_for_library(query: str, library: str, n_results: int = N_RETRIEVE) -> list[dict]:
    """Search ChromaDB for docs about `library` that match `query`."""

    # Embed the query with the same model used during indexing
    query_vec = embedder.encode(query, normalize_embeddings=True).tolist()

    # Retrieve from ChromaDB, filtered by library name
    results = collection.query(
        query_embeddings=[query_vec],
        n_results=n_results,
        where={"library": library},
        include=["documents", "metadatas", "distances"],
    )

    # Package into simple dicts
    docs = []
    for i, text in enumerate(results["documents"][0]):
        docs.append({
            "text":     text,
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        })
    return docs


@traceable(name="rerank_and_filter")
def rerank_and_filter(query: str, docs: list[dict], top_k: int = N_RERANK) -> list[dict]:
    """Score each doc with the cross-encoder, keep the top_k above threshold."""

    if not docs:
        return []

    # Cross-encoder expects (query, document) pairs
    pairs  = [[query, d["text"]] for d in docs]
    scores = reranker.predict(pairs)

    for doc, score in zip(docs, scores):
        doc["rerank_score"] = float(score)

    # Sort descending and apply score gate
    ranked = sorted(docs, key=lambda d: d["rerank_score"], reverse=True)
    return [d for d in ranked if d["rerank_score"] >= MIN_RERANKER_SCORE][:top_k]


@traceable(name="build_rag_context")
def build_rag_context(error_text: str, libraries: list[str]) -> tuple[str, list[dict]]:
    """Retrieve + rerank across all libraries, return combined context string."""

    # Use focused error signal instead of raw traceback for better retrieval
    query    = extract_error_signal(error_text)[:MAX_QUERY]
    all_docs = []

    for lib in libraries:
        raw_docs = retrieve_for_library(query, lib)
        reranked = rerank_and_filter(query, raw_docs)
        all_docs.extend(reranked)

    # Keep best-scoring docs first
    all_docs.sort(key=lambda d: d["rerank_score"], reverse=True)

    # Join texts up to the character budget
    parts, total = [], 0
    for doc in all_docs:
        if total + len(doc["text"]) > MAX_CTX_CHARS:
            break
        parts.append(doc["text"])
        total += len(doc["text"])

    context_str = "\n\n---\n\n".join(parts)
    return context_str, all_docs


# Quick test
_test = retrieve_for_library("SettingWithCopyWarning", "pandas", n_results=2)
print(f"Test: retrieved {len(_test)} docs for pandas")

Test: retrieved 2 docs for pandas


## 8 — Document Summarization (OpenRouter)

Use a free open-source model via **OpenRouter** to summarize retrieved & reranked documentation into **max 5 concise bullet-point hints** before appending them to the user prompt.

In [13]:
# OpenRouter client (used only for summarization — free tier)
# ⚠ OpenRouter REQUIRES HTTP-Referer for free models; without it responses are None.
openrouter_client = openai.OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://github.com/MentorApp",   # required by OpenRouter
        "X-Title":      "MentorApp-RAG",                   # recommended
    },
)
print(f"✓ OpenRouter client ready  ({OPENROUTER_MODEL})")

# Prompt that asks for ≤ 5 bullet hints — includes the error for relevance filtering
SUMMARIZE_PROMPT = """You are a concise technical assistant.
Given the runtime error and documentation snippets below, produce a
MAXIMUM of 5 bullet-point hints (use "•" as bullet character).
Each hint MUST be directly relevant to the specific error shown.
Focus on: correct API usage, required parameters, common pitfalls.

Runtime error:
{error}

Documentation:
{docs}

Rules:
- Return ONLY bullet points ("•"). No intro, no conclusion.
- If the documentation is relevant, extract up to 5 actionable hints.
- If the documentation does NOT address the error, instead write
  exactly 2 short hints from your own knowledge to help fix the error."""

# Track the last API call time for throttling
_last_summarize_call = 0.0


def _local_bullet_fallback(docs_text: str, max_bullets: int = 5) -> str:
    """Last-resort fallback: extract informative sentences as bullet hints."""
    sentences = re.split(r'(?<=[.!?])\s+', docs_text.strip())
    keywords = re.compile(
        r"parameter|argument|return|raise|error|exception|deprecated|default|must|should|require|instead",
        re.IGNORECASE,
    )
    scored = []
    for s in sentences:
        s = s.strip()
        if len(s) < 20 or len(s) > 300:
            continue
        scored.append((len(keywords.findall(s)), s))

    scored.sort(key=lambda x: x[0], reverse=True)
    seen, bullets = set(), []
    for _, s in scored:
        if s not in seen:
            seen.add(s)
            bullets.append(f"• {s}")
        if len(bullets) >= max_bullets:
            break
    if not bullets:
        for s in sentences[:max_bullets]:
            s = s.strip()
            if s:
                bullets.append(f"• {s}")
    return "\n".join(bullets)


def _try_summarize_with_model(model: str, prompt: str) -> str | None:
    """Call a single OpenRouter model. Returns bullet string or None."""
    global _last_summarize_call

    # ── Throttle: respect minimum gap between API calls ───────────────────
    elapsed = time.time() - _last_summarize_call
    if elapsed < SUMMARIZE_THROTTLE:
        time.sleep(SUMMARIZE_THROTTLE - elapsed)

    delay = SUMMARIZE_RETRY_DELAY

    for attempt in range(1, SUMMARIZE_MAX_RETRIES + 1):
        try:
            _last_summarize_call = time.time()
            resp = openrouter_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You summarize documentation into concise bullet points that are directly relevant to a specific error."},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=400,
                temperature=0.0,
            )

            # ── Diagnostic: inspect the raw response ─────────────────────
            choice = resp.choices[0] if resp.choices else None
            if choice is None:
                print(f"    [WARN] {model}: resp.choices is empty (attempt {attempt})")
                if attempt < SUMMARIZE_MAX_RETRIES:
                    time.sleep(delay); delay *= 2
                    continue
                return None

            content = choice.message.content

            # Some models put output in 'reasoning_content' instead
            if content is None and hasattr(choice.message, "reasoning_content"):
                content = getattr(choice.message, "reasoning_content", None)

            if content is None:
                finish = getattr(choice, "finish_reason", "unknown")
                print(f"    [WARN] {model} returned None content "
                      f"(finish_reason={finish}, attempt {attempt}/{SUMMARIZE_MAX_RETRIES})")
                if attempt < SUMMARIZE_MAX_RETRIES:
                    time.sleep(delay); delay *= 2
                    continue
                return None

            summary = content.strip()
            # Accept bullets with "•", "-", or "*"
            bullets = [
                l.strip() for l in summary.split("\n")
                if l.strip() and l.strip()[0] in ("•", "-", "*")
            ]
            if bullets:
                normalized = [("• " + b.lstrip("•-* ")) for b in bullets[:5]]
                return "\n".join(normalized)
            # Model responded but without bullet format — still usable
            return summary[:600]

        except Exception as e:
            print(f"    [WARN] {model} error (attempt {attempt}/{SUMMARIZE_MAX_RETRIES}): {e}")
            if attempt < SUMMARIZE_MAX_RETRIES:
                time.sleep(delay); delay *= 2
            else:
                return None

    return None


@traceable(name="summarize_rag_docs")
def summarize_rag_docs(docs_text: str, error_text: str = "") -> str:
    """Summarize retrieved docs into ≤ 5 bullet hints via OpenRouter.

    When docs_text is empty (no docs retrieved), the summariser is still
    called with the error alone so it can generate 2 fallback hints from
    its own knowledge.

    Strategy to maximize LLM-quality bullets:
    1. Trim docs to SUMMARIZE_MAX_DOC_CHARS (less input → more reliable output)
    2. Throttle calls (SUMMARIZE_THROTTLE seconds gap) to avoid rate limits
    3. Retry with exponential backoff on None responses
    4. If primary model fails, try each OPENROUTER_FALLBACK_MODELS in order
    5. Local sentence extraction only as absolute last resort
    """
    error_signal = extract_error_signal(error_text) if error_text else "Unknown error"
    docs_for_prompt = docs_text[:SUMMARIZE_MAX_DOC_CHARS].strip() if docs_text else "No documentation retrieved."
    prompt = SUMMARIZE_PROMPT.format(
        error=error_signal[:500],
        docs=docs_for_prompt,
    )

    # Try primary model first, then fallbacks
    models_to_try = [OPENROUTER_MODEL] + OPENROUTER_FALLBACK_MODELS

    for model in models_to_try:
        result = _try_summarize_with_model(model, prompt)
        if result is not None:
            return result
        print(f"  [INFO] {model} exhausted — trying next model …")

    # All models failed → local extraction as last resort
    print("  [INFO] All OpenRouter models failed — using local bullet extraction")
    if docs_text.strip():
        return _local_bullet_fallback(docs_text)
    return ""


@traceable(name="build_rag_context_with_summary")
def build_rag_context_with_summary(
    error_text: str,
    libraries: list[str],
    summarize: bool = True,
) -> tuple[str, str, list[dict]]:
    """Retrieve → rerank → (optionally) summarize into bullet hints.
    
    When no docs are retrieved, still calls the summariser so it can
    generate fallback hints from its own knowledge based on the error.
    """

    raw_context, all_docs = build_rag_context(error_text, libraries)

    if summarize:
        print("  Summarizing via OpenRouter …")
        summarized = summarize_rag_docs(raw_context, error_text)
    elif not raw_context:
        return "", "", all_docs
    else:
        summarized = raw_context

    return raw_context, summarized, all_docs


print("✓ Summarization functions defined")

✓ OpenRouter client ready  (openai/gpt-oss-120b)
✓ Summarization functions defined


## 9 — LangChain Chain Setup

Create a `PromptTemplate` and `LLMChain` connected to the **llama.cpp server** for RAG-augmented code generation.

**Components:**
- **System prompt**: Defines the model's role as a Python bug-fixer with structured output format
- **User prompt**: Contains buggy code, error message, and summarized bullet-point hints from OpenRouter
- **Output format**: Uses `<correct_code>` XML tags for reliable parsing

In [48]:
# ── System prompt (tells the model how to respond) ───────────────────────────
SYSTEM_PROMPT = (
    "You fix Python programs.\n"
    "Input: a traceback tail, buggy Python code, and sometimes hints.\n"
    "Use the traceback to find the failing line in the code, then use the hints to fix that exact line first.\n"
    "If the traceback and hints provide a concrete replacement, apply that replacement directly in the code.\n"
    "Do not return the original code unchanged when the failing line is still present.\n"
    "Preserve unrelated code and only make the minimal edits needed to fix the error.\n"
    "Return ONLY this format:\n"
    "<correct_code>\n"
    "(full corrected python code)\n"
    "</correct_code>\n"
    "IMPORTANT: Output NOTHING before <correct_code>. No explanation, no bullet points, no markdown, no backticks. Do not echo the prompt or the hints."
)


# ── User prompt template (filled per sample) ─────────────────────────────────
# The "--- END OF REFERENCE ---" fence + the prefill "<correct_code>" nudge
# prevent the model from echoing or expanding on the hints.
USER_PROMPT_TEMPLATE = """Fix this Python code based on the runtime error.

Traceback:
{error_message}

Failing line from traceback:
{traceback_line}

Code:
{buggy_code}

Hints (use these to fix the failing line first; do NOT repeat them):
{context}

Editing rules:
1. Find the exact failing line from the traceback inside the code.
2. Apply the hint to that line before making any other edits.
3. If the hint gives a concrete replacement, use that exact replacement in every matching occurrence needed to fix the error.
4. Keep the rest of the file unchanged unless another edit is required for correctness.
5. Do not return the original code unchanged if the failing line still contains the invalid value or API usage.
--- END OF REFERENCE ---

<correct_code>"""

# ── Combine into a single prompt template for LangChain ──────────────────────
RAG_PROMPT_TEMPLATE = f"""{SYSTEM_PROMPT}

{{user_prompt}}"""

prompt_template = PromptTemplate(
    input_variables=["user_prompt"],
    template=RAG_PROMPT_TEMPLATE,
)

# stop= tells the server to stop generating once it emits </correct_code>
rag_chain = prompt_template | llm.bind(stop=["</correct_code>"]) | StrOutputParser()

print("System prompt:")
print(SYSTEM_PROMPT)
print("\n✓ LangChain chain ready")

System prompt:
You fix Python programs.
Input: a traceback tail, buggy Python code, and sometimes hints.
Use the traceback to find the failing line in the code, then use the hints to fix that exact line first.
If the traceback and hints provide a concrete replacement, apply that replacement directly in the code.
Do not return the original code unchanged when the failing line is still present.
Preserve unrelated code and only make the minimal edits needed to fix the error.
Return ONLY this format:
<correct_code>
(full corrected python code)
</correct_code>
IMPORTANT: Output NOTHING before <correct_code>. No explanation, no bullet points, no markdown, no backticks. Do not echo the prompt or the hints.

✓ LangChain chain ready


## 10 — Output Parsing & Similarity

Helper functions to extract Python code from model output and compute similarity to reference solutions.

**Parsing priority:**
1. `<correct_code>` XML tags (preferred format)
2. Markdown code blocks (```python)
3. Raw text fallback

In [51]:
def extract_python_code(text: str) -> str:
    """Pull code from model output.

    Because the prompt already contains '<correct_code>' as a prefill and the
    stop sequence is '</correct_code>', the raw output is typically just the
    code itself.  We still handle the full-tag and markdown-fence cases for
    robustness.
    """
    text = (text or "").strip()

    # 1. Full <correct_code>...</correct_code> tags (in case stop didn't fire)
    m = re.search(r"<correct_code>\s*(.*?)\s*</correct_code>", text, re.DOTALL | re.IGNORECASE)
    if m:
        code = m.group(1).strip()
        code = re.sub(r"^```\w*\s*", "", code)
        code = re.sub(r"\s*```$", "", code)
        return code.strip()

    # 2. Opening tag present but no closing tag (stop sequence ate it)
    m2 = re.search(r"<correct_code>\s*(.*)", text, re.DOTALL | re.IGNORECASE)
    if m2:
        return m2.group(1).strip()

    # 3. Markdown code blocks
    blocks = re.findall(r"```(?:python)?\s*([\s\S]*?)```", text, re.IGNORECASE)
    if blocks:
        return blocks[-1].strip()

    # 4. Raw text (the common case with prefill + stop: output IS the code)
    return text.strip()


def extract_error_type(text: str) -> str:
    """Pull error type from <error_type> tags."""
    m = re.search(r"<error_type>\s*(.*?)\s*</error_type>", (text or ""), re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else ""


def extract_traceback_line(error_text: str) -> str:
    """Extract the concrete source line shown in the traceback, if present."""
    lines = (error_text or "").splitlines()
    for i, line in enumerate(lines):
        if line.lstrip().startswith('File '):
            if i + 1 < len(lines):
                candidate = lines[i + 1].strip()
                if candidate and not candidate.startswith(("File ", "Traceback")):
                    return candidate
    return ""


def calculate_similarity(code1: str, code2: str) -> float:
    """Character-level similarity between two code strings (0–1)."""
    if not code1 or not code2:
        return 0.0
    return SequenceMatcher(None, code1, code2).ratio()


def get_libraries_from_sample(sample: dict) -> list[str]:
    """Return library names from sample metadata, or infer from imports."""
    if "libraries" in sample:
        libs = sample["libraries"]
        return [libs] if isinstance(libs, str) else list(libs)

    # Infer from the buggy code (field is 'incorrect_code' in this dataset)
    code = sample.get("incorrect_code", "")
    known = ["pandas", "numpy", "sklearn", "matplotlib", "tensorflow", "keras", "torch"]
    found = [lib for lib in known if lib in code]
    return found or ["pandas"]


# Quick test
_sample_out = "<correct_code>\nimport pandas as pd\nprint('ok')\n</correct_code>"
print("Extraction test:", extract_python_code(_sample_out)[:60])

# Test: raw output when stop sequence fires (no tags in output)
_sample_raw = "import pandas as pd\ndf = pd.read_csv('data.csv')\nprint(df.head())"
print("Raw extraction test:", extract_python_code(_sample_raw)[:60])

Extraction test: import pandas as pd
print('ok')
Raw extraction test: import pandas as pd
df = pd.read_csv('data.csv')
print(df.he


## 11 — RAG Inference Loop

Process each failed sample through the RAG pipeline:
1. Extract libraries from sample
2. Build RAG context from error message (retrieve + rerank)
3. **Summarize** reranked docs into ≤ 5 bullet hints via OpenRouter
4. Run the LangChain chain via llama.cpp server
5. Extract corrected code from `<correct_code>` tags

In [58]:
results = []
total   = len(rag_samples)

print(f"Processing {total} samples …\n" + "=" * 70)

for idx, sample in enumerate(rag_samples):
    sid          = sample["_sid"]
    eval_folder  = sample["_eval_folder"]
    print(f"\n[{idx+1}/{total}] {eval_folder}")

    # ── The "buggy code" is the SFT model's failed prediction ─────────────
    buggy_code    = sample["sft_prediction"]
    # ── The "error message" is the stderr from the smoke test ─────────────
    error_message = sample["runtime_error"]
    traceback_line = extract_traceback_line(error_message)
    # ── Reference correct code from the eval folder's correct.py ──────────
    correct_code  = sample.get("correct_code", "")
    task          = sample.get("title", "")
    libraries     = get_libraries_from_sample({"incorrect_code": buggy_code})

    # ── RAG: retrieve → rerank → summarize ────────────────────────────────
    raw_ctx, summary_ctx, docs = build_rag_context_with_summary(
        error_message, libraries, summarize=True,
    )
    print(f"  docs={len(docs)}  raw={len(raw_ctx)}ch  summary={len(summary_ctx)}ch")
    # if traceback_line:
      #  print(f"  traceback line: {traceback_line}")

    # ── Build prompt & call LLM ───────────────────────────────────────────
    user_prompt = USER_PROMPT_TEMPLATE.format(
        buggy_code=buggy_code,
        error_message=error_message,
        traceback_line=traceback_line or "<not found>",
        context=summary_ctx or "No relevant documentation found.",
    )

    # Full prompt (system + user) as sent to the coder model
    full_input_prompt = RAG_PROMPT_TEMPLATE.format(user_prompt=user_prompt)

    try:
        raw_output = rag_chain.invoke({"user_prompt": user_prompt})
    except Exception as e:
        print(f"  ERROR: {e}")
        raw_output = ""

    predicted   = extract_python_code(raw_output)
    sim = calculate_similarity(predicted, correct_code)
    print(f"  similarity={sim:.3f}")

    # ── Save ──────────────────────────────────────────────────────────────
    # Extract folder index for downstream comparison
    folder_idx = int(eval_folder.split("_")[0]) if eval_folder[0].isdigit() else -1

    result = dict(
        sid=sid, eval_folder=eval_folder, title=task, libraries=libraries,
        buggy_code=buggy_code, error_message=error_message, correct_code=correct_code,
        raw_rag_context=raw_ctx, summarized_context=summary_ctx,
        raw_output=raw_output, predicted_code=predicted,
        sim_to_ref=sim, n_docs_retrieved=len(docs),
        _orig_idx=folder_idx,
    )
    results.append(result)

    out_dir = OUT_DIR / eval_folder
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "predicted.py").write_text(predicted, encoding="utf-8")
    (out_dir / "input_prompt.txt").write_text(user_prompt, encoding="utf-8")
    (out_dir / "raw_output.txt").write_text(raw_output, encoding="utf-8")
    meta = {k: v for k, v in result.items() if k not in ("predicted_code", "buggy_code", "correct_code")}
    (out_dir / "metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("\n" + "=" * 70)
print(f"✓ Done — {len(results)} samples processed")

Processing 16 samples …

[1/16] 016_IMDB_Sentiment_Analysis_Naive_Bayes
  Summarizing via OpenRouter …
  docs=6  raw=2769ch  summary=251ch
  similarity=0.998

[2/16] 017_Reuters_News_Topic_Classification
  Summarizing via OpenRouter …
  docs=6  raw=2557ch  summary=128ch
  similarity=1.000

[3/16] 025_Diabetes_Progression_LightGBM_Regression
  Summarizing via OpenRouter …
  docs=9  raw=2917ch  summary=225ch
  similarity=1.000

[4/16] 026_Titanic_Survival_ROC_Curve
  Summarizing via OpenRouter …
  docs=9  raw=1931ch  summary=250ch
  similarity=0.928

[5/16] 033_Digits_Autoencoder_Reconstruction
  Summarizing via OpenRouter …
  docs=9  raw=3007ch  summary=300ch
  similarity=1.000

[6/16] 046_Penguins_Migration_Time_Series_ARIMA
  Summarizing via OpenRouter …
  docs=9  raw=3007ch  summary=294ch
  similarity=1.000

[7/16] 047_Heart_Model_Calibration_Plot
  Summarizing via OpenRouter …
  docs=9  raw=2988ch  summary=214ch
  similarity=1.000

[8/16] 048_Sunspots_SARIMAX_Seasonal_Forecast_Model

## 12 — Save Summary Results

Export all results to a JSON file for analysis.

In [60]:
summary_path = OUT_DIR / "summary_rag_sft.json"

summary_data = {
    "model": HF_MODEL_REPO,
    "total_samples": len(results),
    "timestamp": datetime.now().isoformat(),
    "config": dict(n_ctx=N_CTX, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                   n_retrieve=N_RETRIEVE, n_rerank=N_RERANK,
                   min_reranker_score=MIN_RERANKER_SCORE, max_ctx_chars=MAX_CTX_CHARS),
    "results": results,
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2)

print(f"✓ Saved {len(results)} results → {summary_path}")

✓ Saved 16 results → /workspace/MentorApp/RAG_Pipelines/RAG_outputs/RAG_with_SFT/summary_rag_sft.json


## 13 — Results Summary

Display statistics about the RAG inference results.

In [61]:
sims      = [r["sim_to_ref"] for r in results]
parseable = [r for r in results if r["predicted_code"].strip()]

print("=" * 50)
print("RAG INFERENCE SUMMARY")
print("=" * 50)
print(f"Samples   : {len(results)}")
print(f"Parseable : {len(parseable)}")

if sims:
    print(f"Similarity: mean={sum(sims)/len(sims):.3f}  min={min(sims):.3f}  max={max(sims):.3f}")
    high = sum(s >= 0.8 for s in sims)
    med  = sum(0.5 <= s < 0.8 for s in sims)
    low  = sum(s < 0.5 for s in sims)
    print(f"  ≥0.8: {high}   0.5–0.8: {med}   <0.5: {low}")

RAG INFERENCE SUMMARY
Samples   : 16
Parseable : 16
Similarity: mean=0.994  min=0.928  max=1.000
  ≥0.8: 16   0.5–0.8: 0   <0.5: 0


---

## 14 — Evaluation: Syntax Check

Verify that generated code compiles without syntax errors using `py_compile`.

In [62]:
def check_syntax(code: str) -> tuple[bool, str]:
    """Return (True, '') if code compiles, else (False, error_msg)."""
    if not code.strip():
        return False, "Empty code"
    try:
        compile(code, "<string>", "exec")
        return True, ""
    except SyntaxError as e:
        return False, f"Line {e.lineno}: {e.msg}"

syntax_results = []
for r in results:
    ok, err = check_syntax(r["predicted_code"])
    syntax_results.append({"sid": r["sid"], "valid": ok, "error": err})

valid_count = sum(s["valid"] for s in syntax_results)
print(f"Syntax valid: {valid_count}/{len(syntax_results)}")

for s in syntax_results:
    if not s["valid"]:
        print(f"  ✗ {s['sid']}: {s['error']}")

Syntax valid: 16/16


## 15 — Fast Eval Patching

Reduce epochs and iterations for faster runtime testing.

In [63]:
NO_EPOCH_PATCH = {"081"}   # samples that must keep original epoch count

def patch_fast_eval(code: str, skip_epoch_patch: bool = False) -> str:
    """Reduce epochs / verbosity so smoke tests finish quickly."""
    if not skip_epoch_patch:
        code = re.sub(r'epochs\s*=\s*\d+',   'epochs=5',    code)
        code = re.sub(r'n_iter\s*=\s*\d+',   'n_iter=5',    code)
        code = re.sub(r'max_iter\s*=\s*\d+', 'max_iter=100', code)

    code = re.sub(r'verbose\s*=\s*[12]', 'verbose=0', code)

    if FORCE_CPU and "CUDA_VISIBLE_DEVICES" not in code:
        code = 'import os\nos.environ["CUDA_VISIBLE_DEVICES"] = ""\n' + code
    return code

# Quick demo
print(patch_fast_eval("model.fit(X, y, epochs=100, verbose=1)"))

model.fit(X, y, epochs=5, verbose=0)


## 16 — Runtime Smoke Test

Run each generated script in an isolated subprocess to check for runtime errors.

In [64]:
def run_script(code: str, timeout: int = TIMEOUT) -> dict:
    """Run code in a subprocess, return status + stdout/stderr + runtime."""
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False, encoding="utf-8") as f:
        f.write(code)
        tmp = f.name
    start = time.time()
    try:
        r = subprocess.run([sys.executable, tmp],
                           capture_output=True, text=True,
                           timeout=timeout, cwd=str(OUT_DIR))
        dt = time.time() - start
        return {"status": "pass" if r.returncode == 0 else "runtime_error",
                "stdout": r.stdout[:2000], "stderr": r.stderr[:2000], "runtime": dt}
    except subprocess.TimeoutExpired:
        return {"status": "timeout", "stdout": "", "stderr": f"Timeout {timeout}s", "runtime": timeout}
    except Exception as e:
        return {"status": "error", "stdout": "", "stderr": str(e), "runtime": time.time() - start}
    finally:
        try: os.unlink(tmp)
        except OSError: pass


# Run smoke tests
valid_samples = [r for r, s in zip(results, syntax_results) if s["valid"]]
smoke_results = []

print(f"Smoke-testing {len(valid_samples)} samples …\n" + "=" * 60)

for i, r in enumerate(valid_samples):
    sid = r["sid"]
    idx_str = sid.split("_")[0]           # e.g. "081"
    patched = patch_fast_eval(r["predicted_code"], skip_epoch_patch=(idx_str in NO_EPOCH_PATCH))

    res = run_script(patched, timeout= 100)
    smoke_results.append({"sid": sid, "_orig_idx": r.get("_orig_idx", -1), **res})

    icon = "✓" if res["status"] == "pass" else "✗"
    print(f"[{i+1}/{len(valid_samples)}] {icon} {sid}  {res['status']}  ({res['runtime']:.1f}s)")
    if res["status"] != "pass" and res["stderr"]:
        last_line = res["stderr"].strip().split("\n")[-1]
        print(f"        {last_line[:100]}")

print("=" * 60)

Smoke-testing 16 samples …
[1/16] ✓ 016_IMDB_Sentiment_Analysis_Naive_Bayes  pass  (14.8s)
[2/16] ✗ 017_Reuters_News_Topic_Classification  runtime_error  (6.9s)
        2026-03-06 14:55:33.163526: W tensorflow/compiler/mlir/tools/
[3/16] ✗ 025_Diabetes_Progression_LightGBM_Regression  timeout  (100.0s)
        Timeout 100s
[4/16] ✗ 026_Titanic_Survival_ROC_Curve  runtime_error  (1.8s)
        LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, 
[5/16] ✗ 033_Digits_Autoencoder_Reconstruction  runtime_error  (5.7s)
        2026-03-06 14:57:20.631074: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is
[6/16] ✓ 046_Penguins_Migration_Time_Series_ARIMA  pass  (2.6s)
[7/16] ✗ 047_Heart_Model_Calibration_Plot  runtime_error  (2.1s)
        ValueError: The least populated classes in y have only 1 member, which is too few. The minimum numbe
[8/16] ✓ 048_Sunspots_SARIMAX_Seasonal_Forecast_Model  pass  (3.6s)
[9/16] ✗ 057_MNIST

## 17 — Evaluation Summary

Aggregate evaluation results and save smoke test report.

In [65]:
from collections import Counter

counts = Counter(sr["status"] for sr in smoke_results)
passed = counts.get("pass", 0)
total  = len(smoke_results)

print("=" * 50)
print("SMOKE TEST SUMMARY")
print("=" * 50)
for status, n in counts.most_common():
    print(f"  {status}: {n}  ({100*n/total:.0f}%)")
print(f"\nPass rate: {passed}/{total} ({100*passed/total:.0f}%)")

# Save report
smoke_out = OUT_DIR / "smoke_report_rag_sft.json"
with open(smoke_out, "w") as f:
    json.dump(smoke_results, f, indent=2)
print(f"✓ Report saved → {smoke_out}")

SMOKE TEST SUMMARY
  runtime_error: 11  (69%)
  pass: 4  (25%)
  timeout: 1  (6%)

Pass rate: 4/16 (25%)
✓ Report saved → /workspace/MentorApp/RAG_Pipelines/RAG_outputs/RAG_with_SFT/smoke_report_rag_sft.json


## 18 — Comparison: SFT vs SFT+RAG

Compare the RAG-enhanced results with the original SFT model pre-test results.

In [67]:
# Compare original SFT pre-test vs SFT + RAG
# smoke_by_orig_idx was built in Cell 12 with _passed flag

total_samples = len(test_set)
orig_passed   = sum(1 for info in smoke_by_orig_idx.values() if info["_passed"])
orig_failed   = total_samples - orig_passed

rag_fixed = sum(
    1 for sr in smoke_results
    if not smoke_by_orig_idx.get(sr["_orig_idx"], {}).get("_passed", True)
    and sr["status"] == "pass"
)
still_failed = sum(
    1 for sr in smoke_results
    if not smoke_by_orig_idx.get(sr["_orig_idx"], {}).get("_passed", True)
    and sr["status"] != "pass"
)
final_passed = orig_passed + rag_fixed

print("=" * 60)
print("SFT  vs  SFT + RAG")
print("=" * 60)
print(f"Dataset size       : {total_samples}")
print(f"SFT passed         : {orig_passed}  ({100*orig_passed/total_samples:.1f}%)")
print(f"SFT failed         : {orig_failed}")
print(f"RAG fixed          : {rag_fixed}")
print(f"Still failed       : {still_failed}")
print(f"Final pass rate    : {final_passed}/{total_samples}  ({100*final_passed/total_samples:.1f}%)")
delta = 100 * final_passed / total_samples - 100 * orig_passed / total_samples
print(f"Improvement        : +{delta:.1f} pp")
print("=" * 60)

SFT  vs  SFT + RAG
Dataset size       : 88
SFT passed         : 70  (79.5%)
SFT failed         : 18
RAG fixed          : 4
Still failed       : 12
Final pass rate    : 74/88  (84.1%)
Improvement        : +4.5 pp
